In [ ]:
!pip install roboflow python-dotenv

import os
from dotenv import load_dotenv
from roboflow import Roboflow

load_dotenv()

DATASET_DIR = "dataset"

if not os.path.exists(os.path.join(DATASET_DIR, "data.yaml")):
    rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
    project = rf.workspace("khalid-abdullah-al-dosari").project("sixray-gzyn7")
    dataset = project.version(1).download("yolov8-obb", location=DATASET_DIR)
else:
    print(f"Dataset already present at ./{DATASET_DIR}, skipping download.")

In [ ]:
import os
import hashlib
import random
from collections import Counter, defaultdict

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from PIL import Image

sns.set_theme(style="whitegrid")
random.seed(42)

with open(os.path.join(DATASET_DIR, "data.yaml")) as f:
    DATA_YAML = yaml.safe_load(f)

CLASS_NAMES = DATA_YAML["names"]
SPLITS = ["train", "valid", "test"]
print("Classes:", CLASS_NAMES)
print("Splits:", SPLITS)

## 1. Dataset Overview

Sixray is an X-ray baggage-screening dataset for prohibited-item detection (oriented bounding boxes). We confirm the source/version, the split sizes, and that every image has a matching label file (no orphans on either side).

In [ ]:
overview_rows = []
image_index = {}  # split -> {stem: image_filename}
label_index = {}  # split -> {stem: label_filename}

for split in SPLITS:
    img_dir = os.path.join(DATASET_DIR, split, "images")
    lbl_dir = os.path.join(DATASET_DIR, split, "labels")

    images = {os.path.splitext(f)[0]: f for f in os.listdir(img_dir)}
    labels = {os.path.splitext(f)[0]: f for f in os.listdir(lbl_dir)}
    image_index[split] = images
    label_index[split] = labels

    images_without_labels = set(images) - set(labels)
    labels_without_images = set(labels) - set(images)

    overview_rows.append({
        "split": split,
        "n_images": len(images),
        "n_labels": len(labels),
        "images_without_labels": len(images_without_labels),
        "labels_without_images": len(labels_without_images),
    })

overview_df = pd.DataFrame(overview_rows).set_index("split")
overview_df.loc["total"] = overview_df.sum()
overview_df

## 2. Parse Labels & Class Distribution

Every label file holds one line per object: `class x1 y1 x2 y2 x3 y3 x4 y4` (normalized OBB corner points). We parse all label files once into a single DataFrame — `objects_df` (one row per annotated object) and `images_df` (one row per image, with its object count) — and reuse both for the rest of the notebook.

In [ ]:
def polygon_area(pts):
    """Shoelace formula for a 4-point polygon in normalized coords."""
    x, y = pts[:, 0], pts[:, 1]
    return 0.5 * abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))


object_rows = []
image_rows = []
malformed_lines = []

for split in SPLITS:
    lbl_dir = os.path.join(DATASET_DIR, split, "labels")
    for stem, lbl_fname in label_index[split].items():
        img_fname = image_index[split].get(stem)
        with open(os.path.join(lbl_dir, lbl_fname)) as f:
            lines = [ln.strip() for ln in f if ln.strip()]

        for line in lines:
            parts = line.split()
            if len(parts) != 9:
                malformed_lines.append((split, lbl_fname, line))
                continue
            cls_id = int(parts[0])
            coords = np.array(parts[1:], dtype=np.float32).reshape(4, 2)
            area = polygon_area(coords)
            out_of_bounds = bool(((coords < -1e-6) | (coords > 1 + 1e-6)).any())
            object_rows.append({
                "split": split,
                "stem": stem,
                "class_id": cls_id,
                "class_name": CLASS_NAMES[cls_id],
                "area": area,
                "degenerate": area < 1e-6 or out_of_bounds,
            })

        image_rows.append({
            "split": split,
            "stem": stem,
            "image_filename": img_fname,
            "n_objects": len(lines),
        })

objects_df = pd.DataFrame(object_rows)
images_df = pd.DataFrame(image_rows)

print(f"Malformed label lines: {len(malformed_lines)}")
print(f"Total annotated objects: {len(objects_df)}")
objects_df.head()

In [ ]:
class_split_counts = (
    objects_df.groupby(["class_name", "split"]).size().unstack(fill_value=0)[SPLITS]
)
class_split_counts["total"] = class_split_counts.sum(axis=1)
class_split_counts = class_split_counts.sort_values("total", ascending=False)
display(class_split_counts)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

class_split_counts[SPLITS].plot(kind="bar", stacked=True, ax=axes[0])
axes[0].set_title("Object instances per class, by split")
axes[0].set_xlabel("class")
axes[0].set_ylabel("instance count")
axes[0].tick_params(axis="x", rotation=45)

imbalance_ratio = class_split_counts["total"].max() / class_split_counts["total"].min()
sns.barplot(
    x=class_split_counts.index, y=class_split_counts["total"], ax=axes[1], hue=class_split_counts.index,
    palette="viridis", legend=False,
)
axes[1].set_title(f"Total instances per class (max/min ratio = {imbalance_ratio:.2f}x)")
axes[1].set_xlabel("class")
axes[1].set_ylabel("instance count")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

print(f"Class imbalance ratio (most frequent / least frequent): {imbalance_ratio:.2f}x")

## 3. Image Property Analysis

X-ray scanner captures rarely come out at a uniform resolution. We check the width/height/aspect-ratio spread to confirm resizing is needed before training (YOLO will letterbox to `imgsz` internally, but it's worth knowing how much variance we're feeding it).

In [ ]:
size_rows = []
corrupt_images = []

for split in SPLITS:
    img_dir = os.path.join(DATASET_DIR, split, "images")
    for stem, fname in image_index[split].items():
        fpath = os.path.join(img_dir, fname)
        try:
            with Image.open(fpath) as im:
                w, h = im.size
        except Exception as e:
            corrupt_images.append((split, fname, str(e)))
            continue
        size_rows.append({"split": split, "stem": stem, "width": w, "height": h, "aspect_ratio": w / h})

sizes_df = pd.DataFrame(size_rows)
print(f"Corrupt/unreadable images: {len(corrupt_images)}")
sizes_df[["width", "height", "aspect_ratio"]].describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(sizes_df["width"], bins=40, ax=axes[0], color="steelblue")
axes[0].set_title("Image width (px)")
sns.histplot(sizes_df["height"], bins=40, ax=axes[1], color="darkorange")
axes[1].set_title("Image height (px)")
sns.histplot(sizes_df["aspect_ratio"], bins=40, ax=axes[2], color="seagreen")
axes[2].set_title("Aspect ratio (w / h)")
plt.tight_layout()
plt.show()

print(f"Unique resolutions: {sizes_df[['width', 'height']].drop_duplicates().shape[0]} out of {len(sizes_df)} images")
print("Most common resolutions:")
print(sizes_df.groupby(["width", "height"]).size().sort_values(ascending=False).head())

## 4. Sample Visualization (OBB Annotations)

Drawing the oriented bounding boxes on top of real samples is the fastest way to sanity-check that labels line up with objects and that the class names make sense.

In [ ]:
CLASS_COLORS = {
    "Gun": (255, 0, 0), "Knife": (0, 255, 0), "Pliers": (0, 128, 255),
    "Scissors": (255, 0, 255), "Wrench": (255, 165, 0),
}


def load_obb_labels(label_path):
    boxes = []
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            cls_id = int(parts[0])
            coords = np.array(parts[1:], dtype=np.float32).reshape(4, 2)
            boxes.append((CLASS_NAMES[cls_id], coords))
    return boxes


def draw_obb(image_bgr, boxes):
    image_bgr = image_bgr.copy()
    h, w = image_bgr.shape[:2]
    for class_name, coords in boxes:
        pts = (coords * [w, h]).astype(np.int32)
        color = CLASS_COLORS.get(class_name, (200, 200, 200))
        cv2.polylines(image_bgr, [pts], isClosed=True, color=color, thickness=2)
        cv2.putText(image_bgr, class_name, tuple(pts[0]), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    return image_bgr


sample_stems = random.sample(list(image_index["train"]), 6)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, stem in zip(axes.flat, sample_stems):
    img_path = os.path.join(DATASET_DIR, "train", "images", image_index["train"][stem])
    lbl_path = os.path.join(DATASET_DIR, "train", "labels", label_index["train"][stem])
    img = cv2.imread(img_path)
    boxes = load_obb_labels(lbl_path)
    drawn = cv2.cvtColor(draw_obb(img, boxes), cv2.COLOR_BGR2RGB)
    ax.imshow(drawn)
    ax.set_title(f"{stem} ({len(boxes)} objects)")
    ax.axis("off")

plt.tight_layout()
plt.show()

## 5. Data Quality Checks

Per the project brief: check quality, remove duplicates/unsuitable samples, and fix inaccurate labels. We check for exact-duplicate images (including **cross-split leakage**, which would silently inflate validation/test scores), empty-label ("background") images, and degenerate or out-of-bounds OBB polygons (already flagged while parsing in section 2).

In [ ]:
# --- Exact-duplicate / cross-split leakage check (MD5 of raw image bytes) ---
hash_to_locations = defaultdict(list)
for split in SPLITS:
    img_dir = os.path.join(DATASET_DIR, split, "images")
    for stem, fname in image_index[split].items():
        with open(os.path.join(img_dir, fname), "rb") as f:
            file_hash = hashlib.md5(f.read()).hexdigest()
        hash_to_locations[file_hash].append((split, stem))

duplicate_groups = {h: locs for h, locs in hash_to_locations.items() if len(locs) > 1}
cross_split_leaks = {h: locs for h, locs in duplicate_groups.items() if len({s for s, _ in locs}) > 1}

# --- Empty-label ("background") images ---
empty_label_images = images_df[images_df["n_objects"] == 0]

# --- Degenerate / out-of-bounds boxes (flagged during parsing) ---
degenerate_boxes = objects_df[objects_df["degenerate"]]

print("=== Data Quality Report ===")
print(f"Corrupt/unreadable images:        {len(corrupt_images)}")
print(f"Malformed label lines:             {len(malformed_lines)}")
print(f"Exact-duplicate image groups:      {len(duplicate_groups)}")
print(f"  of which cross-split (leakage):  {len(cross_split_leaks)}")
print(f"Empty-label ('background') images: {len(empty_label_images)} ({empty_label_images.groupby('split').size().to_dict()})")
print(f"Degenerate/out-of-bounds boxes:    {len(degenerate_boxes)}")

print("\nObjects-per-image distribution:")
display(images_df.groupby("split")["n_objects"].describe())

## 6. Class Balance Across Splits

Beyond overall class imbalance (section 2), we check that train/valid/test each carry roughly the same class *proportions* — if one split skews toward a different class mix than the others, evaluation metrics won't be representative.

In [ ]:
class_proportions = class_split_counts[SPLITS].div(class_split_counts[SPLITS].sum(axis=0), axis=1)
display(class_proportions.style.format("{:.1%}"))

class_proportions.plot(kind="bar", figsize=(9, 5))
plt.title("Class proportion within each split (should track closely across splits)")
plt.ylabel("share of split's objects")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

max_spread = (class_proportions.max(axis=1) - class_proportions.min(axis=1)).max()
print(f"Largest cross-split proportion spread for a single class: {max_spread:.1%}")

## 7. Preprocessing — Data Cleaning

Acts on section 5's findings: drop any corrupt images, drop exact-duplicate images (keeping one copy, prioritizing `train`), and drop degenerate/out-of-bounds object rows. `clean_stems` below is the per-split set of image stems that pass all checks — everything downstream (training) should read through this list rather than the raw folder listing, so unresolved issues can't silently leak in.

In [ ]:
corrupt_set = {(split, os.path.splitext(fname)[0]) for split, fname, _ in corrupt_images}

duplicate_drop_set = set()
for locs in duplicate_groups.values():
    # keep the copy from the highest-priority split (train > valid > test), drop the rest
    locs_sorted = sorted(locs, key=lambda sl: SPLITS.index(sl[0]))
    duplicate_drop_set.update(locs_sorted[1:])

clean_stems = {
    split: [stem for stem in image_index[split]
            if (split, stem) not in corrupt_set and (split, stem) not in duplicate_drop_set]
    for split in SPLITS
}
clean_stem_sets = {split: set(stems) for split, stems in clean_stems.items()}

clean_objects_df = objects_df[~objects_df["degenerate"]].copy()
clean_objects_df = clean_objects_df[
    clean_objects_df.apply(lambda r: r["stem"] in clean_stem_sets[r["split"]], axis=1)
]

for split in SPLITS:
    dropped = len(image_index[split]) - len(clean_stems[split])
    print(f"{split}: {len(clean_stems[split])} images kept, {dropped} dropped (corrupt/duplicate)")

dropped_objects = len(objects_df) - len(clean_objects_df)
print(f"\nObjects dropped as degenerate/out-of-bounds: {dropped_objects}")

## 8. Preprocessing — Data Augmentation

Ultralytics' YOLOv8-OBB trainer already applies on-the-fly augmentation (mosaic, HSV jitter, flips, rotation/translation/scale/shear) during `model.train()`, so we don't need to pre-bake an augmented copy of the dataset to disk. What's demonstrated here instead:

1. An OBB-safe augmentation transform (rotation + horizontal flip applied consistently to both the image *and* the polygon corner points — unlike axis-aligned boxes, arbitrary rotation angles are fine for OBB).
2. A per-image **sample weight** for training, upweighting images that contain the rarer classes (Scissors, Knife — roughly a 4-5x imbalance vs. Pliers/Gun, see section 2), so training sees them more often without needing to synthesize new images.

In [ ]:
def augment_obb(image_bgr, boxes, angle_deg, flip_h):
    """Rotate (+ optional horizontal flip) an image and its OBB polygons together."""
    h, w = image_bgr.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle_deg, 1.0)
    rotated = cv2.warpAffine(image_bgr, M, (w, h), borderMode=cv2.BORDER_REFLECT)

    new_boxes = []
    for class_name, coords in boxes:
        pts = coords * [w, h]
        pts_h = np.hstack([pts, np.ones((4, 1))])
        rot_pts = (M @ pts_h.T).T
        if flip_h:
            rot_pts[:, 0] = w - rot_pts[:, 0]
        new_boxes.append((class_name, (rot_pts / [w, h]).astype(np.float32)))

    if flip_h:
        rotated = cv2.flip(rotated, 1)
    return rotated, new_boxes


demo_stem = sample_stems[0]
demo_img = cv2.imread(os.path.join(DATASET_DIR, "train", "images", image_index["train"][demo_stem]))
demo_boxes = load_obb_labels(os.path.join(DATASET_DIR, "train", "labels", label_index["train"][demo_stem]))
aug_img, aug_boxes = augment_obb(demo_img, demo_boxes, angle_deg=15, flip_h=True)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].imshow(cv2.cvtColor(draw_obb(demo_img, demo_boxes), cv2.COLOR_BGR2RGB))
axes[0].set_title("Original")
axes[0].axis("off")
axes[1].imshow(cv2.cvtColor(draw_obb(aug_img, aug_boxes), cv2.COLOR_BGR2RGB))
axes[1].set_title("Augmented (rotate 15°, h-flip)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Inverse-frequency class weight, then each training image's weight = max over the
# classes it contains (an image with a rare class gets sampled more often).
class_freq = clean_objects_df[clean_objects_df["split"] == "train"]["class_name"].value_counts()
inv_freq_weight = (class_freq.max() / class_freq).to_dict()

train_objects = clean_objects_df[clean_objects_df["split"] == "train"]
stem_to_weight = (
    train_objects.groupby("stem")["class_name"]
    .apply(lambda names: max(inv_freq_weight[n] for n in names))
)
# background (empty-label) train images keep a neutral weight of 1.0
sample_weights = pd.Series(1.0, index=clean_stems["train"])
sample_weights.update(stem_to_weight)
sample_weights.name = "sample_weight"

print("Inverse-frequency class weights (train split):")
print(pd.Series(inv_freq_weight).sort_values(ascending=False))
print(f"\nPer-image sample weights computed for {len(sample_weights)} training images")
sample_weights.sort_values(ascending=False).head()

## 9. Finalize & Export

Apply the section 7 cleaning decisions to disk (moving flagged files into a `_removed/` quarantine per split, rather than deleting — reversible if a check turns out to be a false positive), and export the training sample weights from section 8 so the training notebook can just load them.

In [ ]:
import shutil

to_remove = corrupt_set | duplicate_drop_set
for split, stem in to_remove:
    quarantine_img = os.path.join(DATASET_DIR, split, "_removed", "images")
    quarantine_lbl = os.path.join(DATASET_DIR, split, "_removed", "labels")
    os.makedirs(quarantine_img, exist_ok=True)
    os.makedirs(quarantine_lbl, exist_ok=True)

    img_fname = image_index[split].get(stem)
    lbl_fname = label_index[split].get(stem)
    if img_fname:
        src = os.path.join(DATASET_DIR, split, "images", img_fname)
        if os.path.exists(src):
            shutil.move(src, os.path.join(quarantine_img, img_fname))
    if lbl_fname:
        src = os.path.join(DATASET_DIR, split, "labels", lbl_fname)
        if os.path.exists(src):
            shutil.move(src, os.path.join(quarantine_lbl, lbl_fname))

print(f"Quarantined {len(to_remove)} image/label pairs (corrupt or duplicate).")

sample_weights.to_csv(os.path.join(DATASET_DIR, "train_sample_weights.csv"), header=True)
print(f"Saved training sample weights to {DATASET_DIR}/train_sample_weights.csv")

## 10. Readiness Checklist

Maps back to the capstone brief's data-processing requirements (section 6 of the project guide).

In [ ]:
checklist = [
    ("Data source documented", "Roboflow project 'sixray-gzyn7' (Sixray X-ray baggage screening), v1"),
    ("Image/class counts described", f"{len(images_df)} images, {len(CLASS_NAMES)} classes: {list(CLASS_NAMES.values())}"),
    ("Quality checked, duplicates removed", f"{len(corrupt_images)} corrupt, {len(duplicate_groups)} duplicate groups ({len(cross_split_leaks)} cross-split) -> quarantined"),
    ("Labels validated", f"{len(malformed_lines)} malformed lines, {len(degenerate_boxes)} degenerate/out-of-bounds boxes -> excluded"),
    ("Train/valid/test split confirmed", f"train={len(clean_stems['train'])}, valid={len(clean_stems['valid'])}, test={len(clean_stems['test'])}, no cross-split leakage"),
    ("Data augmentation strategy defined", "OBB-safe rotate+flip transform demoed; on-the-fly aug via YOLOv8-OBB trainer at train time"),
    ("Class balance examined", f"imbalance ratio {imbalance_ratio:.2f}x (max/min class) -> per-image sample weights exported"),
    ("Samples visualized", "6 annotated samples rendered with OBB polygons in section 4"),
]

checklist_df = pd.DataFrame(checklist, columns=["Requirement", "Status / Evidence"])
checklist_df